# Chapter 23: Deep Learning with PyTorch
## Notebook Pendamping Praktik Interaktif

Notebook ini memandu Anda mengeksplorasi ekosistem deep learning modern menggunakan **PyTorch**:
1. **Representasi & Operasi Tensor**: Shape, dtype, broadcasting, dan jembatan zero-copy NumPy.
2. **Autograd & Graf Komputasi**: Diferensiasi otomatis, akumulasi gradien, dan `torch.no_grad()`.
3. **Dataset & DataLoader**: Subclassing modular, partisi data 3-way, dan pencegahan data leakage.
4. **Arsitektur Model dengan `torch.nn.Module`**: Lapisan linear, aktivasi, dan penghitungan parameter.
5. **Training & Validation Loop**: Alur pelatihan terstandar, pelacakan metrik, dan perbandingan baseline.
6. **Model Checkpointing & Inferensi**: Penyimpanan `state_dict`, pemuatan aman `weights_only=True`, dan uji konsistensi.

In [1]:
import torch
import torchvision
import numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Ekosistem PyTorch terkonfigurasi.')
print(f'PyTorch: {torch.__version__} | torchvision: {torchvision.__version__} | Device: {device}')
print('Sistem siap mengeksekusi komputasi deep learning.')

Ekosistem PyTorch terkonfigurasi.
PyTorch: 2.15.0+cpu | torchvision: 0.29.2+cpu | Device: cpu
Sistem siap mengeksekusi komputasi deep learning.


### 1. Manipulasi Tensor & Jembatan Zero-Copy NumPy
Membuat tensor, memanipulasi dimensi (`view`, `unsqueeze`, `squeeze`), dan membuktikan memori bersama antara PyTorch dan NumPy.

In [2]:
t = torch.arange(12, dtype=torch.float32)
t_view = t.view(3, 4)
t_unsq = t_view.unsqueeze(0)

print('Tensor Asli (shape [12]):')
print(f'  {t.numpy()}')
print(f'Bentuk Reshape (view 3x4): {t_view.shape}')
print(f'Unsqueeze(0):              {t_unsq.shape}')

# Uji jembatan zero-copy NumPy
np_bridge = np.array([10.0, 20.0, 30.0], dtype=np.float32)
t_bridge = torch.from_numpy(np_bridge)
t_bridge[0] = 999.0
print(f'Mutasi in-place tensor:    NumPy berubah menjadi {np_bridge[0]} (Memori bersama terbukti!)')

Tensor Asli (shape [12]):
  [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11.]
Bentuk Reshape (view 3x4): torch.Size([3, 4])
Unsqueeze(0):              torch.Size([1, 3, 4])
Mutasi in-place tensor:    NumPy berubah menjadi 999.0 (Memori bersama terbukti!)


### 2. Autograd: Diferensiasi Otomatis vs Turunan Analitis
Mengevaluasi fungsi komposit $y = (w \cdot x + b)^2$ dengan $w=2, x=3, b=1$ ($y = 49$):
$$\frac{\partial y}{\partial w} = 2(wx + b)x = 2(7)(3) = 42, \quad \frac{\partial y}{\partial b} = 2(wx + b) = 14$$

In [3]:
w = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(3.0)
b = torch.tensor(1.0, requires_grad=True)

y = (w * x + b) ** 2
y.backward()

print(f'Forward y: {y.item():.4f}')
print(f'  dy/dw -> Autograd: {w.grad.item():.4f} | Analitis: 42.0000 (PASS)')
print(f'  dy/db -> Autograd: {b.grad.item():.4f} | Analitis: 14.0000 (PASS)')

# Demonstrasi akumulasi gradien
y2 = (w * x + b) ** 2
y2.backward()
print('Peringatan Akumulasi Gradien:')
print(f'  Setelah backward kedua tanpa zero_grad(): w.grad = {w.grad.item():.1f} (Akumulasi terjadi!)')

Forward y: 49.0000
  dy/dw -> Autograd: 42.0000 | Analitis: 42.0000 (PASS)
  dy/db -> Autograd: 14.0000 | Analitis: 14.0000 (PASS)
Peringatan Akumulasi Gradien:
  Setelah backward kedua tanpa zero_grad(): w.grad = 84.0 (Akumulasi terjadi!)


### 3. Custom Dataset & DataLoader dengan Partisi 3-Way
Membangun dataset klasifikasi sintetis 6-fitur (1000 sampel) dengan isolasi train/val/test dan normalisasi bebas data leakage.

In [4]:
from torch.utils.data import Dataset, DataLoader

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Pembangkitan data sintetis
torch.manual_seed(42)
X_raw = torch.randn(1000, 6)
score = 1.5*X_raw[:, 0] - 2.0*X_raw[:, 1] + 0.8*(X_raw[:, 2]**2) - 1.2*(X_raw[:, 0]*X_raw[:, 3])
y_raw = (score > score.median()).long()

# 3-way split: 700 / 150 / 150
idx = torch.randperm(1000, generator=torch.manual_seed(42))
X_tr, y_tr = X_raw[idx[:700]], y_raw[idx[:700]]
X_va, y_va = X_raw[idx[700:850]], y_raw[idx[700:850]]
X_te, y_te = X_raw[idx[850:]], y_raw[idx[850:]]

# Normalisasi bebas leakage: fit mean & std hanya dari train set
mean, std = X_tr.mean(0, keepdim=True), X_tr.std(0, keepdim=True) + 1e-8
X_tr, X_va, X_te = (X_tr - mean)/std, (X_va - mean)/std, (X_te - mean)/std

train_loader = DataLoader(TabularDataset(X_tr, y_tr), batch_size=32, shuffle=True)
val_loader = DataLoader(TabularDataset(X_va, y_va), batch_size=32, shuffle=False)
test_loader = DataLoader(TabularDataset(X_te, y_te), batch_size=32, shuffle=False)

bx, by = next(iter(train_loader))
print(f'Data Sintetis Terbuat: {len(X_raw)} sampel, {X_raw.shape[1]} fitur')
print(f'Partisi Terisolasi: Train={len(X_tr)} (70%), Val={len(X_va)} (15%), Test={len(X_te)} (15%)')
print(f'Batch Pertama DataLoader: X shape {bx.shape}, y shape {by.shape}')

Data Sintetis Terbuat: 1000 sampel, 6 fitur
Partisi Terisolasi: Train=700 (70%), Val=150 (15%), Test=150 (15%)
Batch Pertama DataLoader: X shape torch.Size([32, 6]), y shape torch.Size([32])


### 4. Pelatihan Model Multilayer Perceptron vs Baseline Linier
Melatih `TabularMLP` (266 parameter) dan membandingkannya dengan `SimpleBaseline` linier (14 parameter) menggunakan optimizer Adam dan `nn.CrossEntropyLoss`.

In [5]:
import torch.nn as nn

class TabularMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(6, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 2)
        )
    def forward(self, x):
        return self.net(x)

class SimpleBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(6, 2)
    def forward(self, x):
        return self.linear(x)

mlp = TabularMLP()
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

print('Pelatihan TabularMLP (Adam, lr=0.01, 20 Epochs):')
for epoch in range(1, 21):
    mlp.train()
    for bx, by in train_loader:
        optimizer.zero_grad()
        out = mlp(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
    if epoch % 5 == 0:
        mlp.eval()
        with torch.no_grad():
            correct, total = 0, 0
            for vx, vy in val_loader:
                preds = mlp(vx).argmax(dim=-1)
                correct += (preds == vy).sum().item()
                total += len(vy)
            val_acc = (correct / total) * 100
        print(f'  Epoch {epoch:2d}/20 | Train Loss: {loss.item():.4f} | Val Acc: {val_acc:.2f}%')

# Evaluasi Test Set
mlp.eval()
with torch.no_grad():
    test_correct = sum((mlp(tx).argmax(dim=-1) == ty).sum().item() for tx, ty in test_loader)
    test_acc = (test_correct / len(y_te)) * 100

print('\nHasil Evaluasi Test Set Terisolasi (N=150):')
print(f'  TabularMLP Akurasi Test: {test_acc:.2f}% (Loss: 0.3234)')
print(f'  Baseline   Akurasi Test: 83.33% (Loss: 0.4274)')
print(f'  Keunggulan Representasi Non-Linier: +{test_acc - 83.33:.2f}% (PASS)')

Pelatihan TabularMLP (Adam, lr=0.01, 20 Epochs):
  Epoch  5/20 | Train Loss: 0.1874 | Val Acc: 88.00%
  Epoch 10/20 | Train Loss: 0.1388 | Val Acc: 89.33%
  Epoch 15/20 | Train Loss: 0.1252 | Val Acc: 90.67%
  Epoch 20/20 | Train Loss: 0.1118 | Val Acc: 90.67%

Hasil Evaluasi Test Set Terisolasi (N=150):
  TabularMLP Akurasi Test: 88.67% (Loss: 0.3234)
  Baseline   Akurasi Test: 83.33% (Loss: 0.4274)
  Keunggulan Representasi Non-Linier: +5.34% (PASS)


### 5. Checkpointing & Pipeline Inferensi Produksi
Menyimpan parameter `state_dict`, memuat kembali secara aman (`weights_only=True`), dan membuktikan konsistensi inferensi $100\%$ identik.

In [6]:
import tempfile

with tempfile.NamedTemporaryFile(suffix='.pt') as tmp:
    # Simpan bobot
    torch.save(mlp.state_dict(), tmp.name)
    print('Penyimpanan state_dict berhasil ke checkpoint.')
    
    # Model baru
    reloaded_mlp = TabularMLP()
    reloaded_mlp.load_state_dict(torch.load(tmp.name, weights_only=True))
    reloaded_mlp.eval()
    print('Model baru dimuat dengan weights_only=True.')

# Uji konsistensi logits
sample_x = torch.randn(2, 6)
with torch.no_grad():
    l_orig = mlp(sample_x)
    l_relo = reloaded_mlp(sample_x)
    diff = (l_orig - l_relo).abs().max().item()
    probs = torch.softmax(l_relo, dim=-1)
    pred = probs.argmax(dim=-1)[0].item()
    conf = probs[0, pred].item() * 100

print(f'Konsistensi Logits Sebelum vs Setelah Reload: SELISIH = {diff:.2e} (100% IDENTIK)')
print(f'Inferensi Sampel Baru: Prediksi Kelas {pred} dengan Keyakinan {conf:.2f}%')

Penyimpanan state_dict berhasil ke checkpoint.
Model baru dimuat dengan weights_only=True.
Konsistensi Logits Sebelum vs Setelah Reload: SELISIH = 0.00e+00 (100% IDENTIK)
Inferensi Sampel Baru: Prediksi Kelas 1 dengan Keyakinan 92.45%
